# mixedbench in a nutshell

Generalized linear mixed models (GLMMs) fit data that come in groups:
repeated measures per subject, pupils per school, plots per field.
Methods that do Bayesian inference on GLMMs are hard to compare,
since each paper tests on its own handful of datasets.
`mixedbench` offers a solution for this: it comes with a registry of public hierarchical tabular datasets with their documented GLMMs. Each dataset is fetched from a pinned source and packaged into an inference problem ready to fit.

This notebook demos our pipeline using the notorious [sleepstudy](https://rdrr.io/cran/lme4/man/sleepstudy.html) (reaction times of 18 subjects over ten days of restricted sleep).

On Colab or in a fresh environment, install mixedbench first:

In [1]:
%pip install -q mixedbench

/Users/alex/.cache/uv/builds-v0/.tmpQa2gng/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


#### Overview:
1. Browse the registry
2. Load one table
3. Build one problem
4. Outlook

## 1. Browse the registry

The registry lists every documented model the survey found on public grouped data, one row per table and model.
A *suite* is a frozen subset of it including reference posteriors to score against.
(Suite 1.1 is the first; its references come with release 1.1, so this notebook works on the registry.)

The package ships the registry files:

In [2]:
from pathlib import Path

import pandas as pd
from mixedbench import registry

sorted(file.name for file in registry.path().iterdir())

['SUMMARY.md',
 'columns.csv',
 'dimensions.csv',
 'excluded.csv',
 'manifest.csv',
 'quantiles.yaml',
 'registry.csv',
 'sources.yaml']

A registry row is a single documented model for a single table. Some tables have multiple documented models. The naming scheme for a row is `<source>/<object>/<model_index>`: the source the table comes from, the table's name in it, and the model's number.

In [3]:
rows = registry.rows()
tables = {(row.source, row.object) for row in rows}
print(f'{len(rows)} documented models over {len(tables)} tables')
columns = ['id', 'response_type', 'formula']
records = [{column: getattr(row, column) for column in columns} for row in rows]
pd.DataFrame(records).head()

714 documented models over 641 tables


,id,response_type,formula
0,lme4/Arabidopsis/1,count,total.fruits ~ nutrient*amd + rack + status + ...
1,lme4/Dyestuff/1,continuous,Yield ~ 1 + (1|Batch)
2,lme4/Pastes/1,continuous,strength ~ (1|batch/cask)
3,lme4/Penicillin/1,continuous,diameter ~ (1|plate) + (1|sample)
4,lme4/VerbAgg/1,binary,r2 ~ (Anger + Gender + btype + situ)^2 + (1|id...


Every table is fetched from a pinned source: a CRAN package at a version, a git repository at a revision, or a file at a URL with its sha256.

In [4]:
sources = registry.sources()
labels = sorted(
    {row.source for row in rows}, key=list(sources).index
)  # in the order of sources.yaml
pinned = [
    {'source': label, 'kind': sources[label].kind, 'pin': sources[label].pin} for label in labels
]
kinds = pd.Series([entry['kind'] for entry in pinned]).value_counts()
print(f'{len(labels)} sources:', kinds.to_dict())
pd.DataFrame(pinned).head()

173 sources: {'url': 142, 'cran': 23, 'git': 8}


,source,kind,pin
0,lme4,cran,2.0-6
1,nlme,cran,3.1-171
2,mlmRev,cran,1.0-9
3,MEMSS,cran,0.9-4
4,glmmTMB,cran,1.1.15.2


## 2. Load one table

The loader downloads the source at its pin into `cache/` (here [lme4](https://rdrr.io/cran/lme4/) from CRAN),
reads the one file that holds the table, checks it against its sha256,
and returns the table with the column types the row declares.

In [5]:
cache = Path('cache')
[row] = [row for row in rows if row.id == 'lme4/sleepstudy/1']
print(row.formula)
table = registry.Tables(cache).load(row)
table.head()

Reaction ~ Days + (Days|Subject)


,Reaction,Days,Subject
0,249.5600,0.0,308
1,258.7047,1.0,308
2,250.8006,2.0,308
3,321.4398,3.0,308
4,356.8519,4.0,308


## 3. Build one problem

A *problem* is a registry row made ready to fit and score:
the table coded into a design matrix, a response and grouping codes,
together with the family and link, the prior, the held-out splits and the scales.
The *contract* is the fixed set of rules that does this, so that two methods fitted to one problem compare.

`Problem.build` builds the problem from the row and its table.
`materialise` writes the problem to two files in the cache and returns its hash,
a fingerprint over every value: the same hash means the same problem on any machine.
`load` reads it back from those files offline.

In [6]:
from mixedbench.contract import Problem, load, materialise

materialise(Problem.build(row, table), cache)

'8111b74ef393568a2cb445ef9f10cc4f538abd8d6a6d48c6b91181fac5a8f330'

In [7]:
problem = load('lme4/sleepstudy/1', cache)
print('design: ', problem.design)
print('response:', problem.response, f'({problem.family}, {problem.link} link)')
print('factors: ', problem.factors)
problem.table.head()

design:  ('Intercept', 'Days')
response: Reaction (gaussian, identity link)
factors:  ('Subject',)


,Intercept,Days,Reaction,Subject,rows,groups
0,1.0,0.0,249.5600,0,seen,seen
1,1.0,1.0,258.7047,0,seen,seen
2,1.0,2.0,250.8006,0,seen,seen
3,1.0,3.0,321.4398,0,seen,seen
4,1.0,4.0,356.8519,0,seen,seen


The coded columns stay in raw units in the table; the problem stores their standardised scales beside it (below).
On that standardised scale, every problem comes with the same prior, one per brms parameter class.
The reference posteriors are drawn under this prior, so a method scored against them uses it too;
a method scored on held-out data may bring its own.

In [8]:
problem.prior

{'Intercept': Prior(distribution='normal', parameters=(0, 2.5)),
 'b': Prior(distribution='normal', parameters=(0, 2.5)),
 'cor': Prior(distribution='lkj', parameters=(1,)),
 'sd': Prior(distribution='student_t', parameters=(3, 0, 2.5)),
 'sigma': Prior(distribution='student_t', parameters=(3, 0, 2.5))}

A method is scored on one of two tracks:
the *posterior track* fits the full table and compares its draws with the reference posterior,
the *predictive track* fits the seen rows of a split and predicts the held-out ones.
There are two splits: held-out rows (new observations of known groups) and held-out groups (new groups).
The rows a fit sees, its *regime* (`full`, `rows`, `groups`), set the mean and standard deviation each column is standardised with:

In [9]:
scales = [
    {'regime': regime, 'column': column, 'mean': scale.mean, 'sd': scale.sd}
    for regime, columns in problem.scales.items()
    for column, scale in columns.items()
]
pd.DataFrame(scales)

,regime,column,mean,sd
0,full,Days,4.500000,2.880293
1,full,Reaction,298.507892,56.328757
2,groups,Days,4.500000,2.881299
3,groups,Reaction,298.563104,58.626440
4,rows,Days,4.425926,2.838960
5,rows,Reaction,298.159565,55.611442


## 5. Outlook

Release 1.1 adds reference posteriors for the problems of suite 1.1,
the metrics that score an approximation against them, and the scores of standard methods to compare with;
this notebook will be updated accordingly.
The plan is in the [roadmap](https://github.com/adkipnis/mixedbench/blob/main/ROADMAP.md).